# Cicero native engine legal order replay

This notebook belongs to the project's sequential measurement and validation programme. Read its result as evidence about behavioural validity, representation, comparator strength, timing, information matching, or mechanistic calibration as appropriate. Legacy H1/H_priv identifiers may remain inside code, saved paths, or frozen schemas for reproducibility; they are not the object being "found" by the current analysis.

**Repository framing.** The current paper separates target-specific representation from predictive privilege. A positive neural result is interpreted only after behavioural validity, control, comparator-strength, timing, and information-set checks.


# Latent reservations — Notebook 11
## CICERO native engine reconstruction + legal-order sibling replay

Notebook 10 established the released full-press state bank and froze the scientific target.

Notebook 11 does not estimate H1. It proves the mechanical substrate for the first local-subject order experiment:

- load/build the official Meta `pydipcc` engine at Notebook 10's exact commit;
- reconstruct released movement phases;
- verify released board state against native reconstruction;
- enumerate focal-power legal orders;
- select two legal focal alternatives before any local-subject outcome;
- clone one identical native parent;
- keep all non-focal orders fixed;
- process both focal-order siblings;
- require different native post-phase consequences;
- freeze focal-power released messages and exact parent text for Notebook 12.

Historical CICERO focal orders remain metadata, never the local subject's label.


### v4 native dependency repair

The previous diagnostics prove that the `pydipcc` target exists and begins compiling.
Compilation stops because `glog/logging.h` is missing.

v4 changes only native dependency/bootstrap handling:

- compile-probe Google glog before the CICERO build;
- install the system glog development package when package-manager privileges are available;
- otherwise build pinned Google glog v0.6.0 into a project-local prefix;
- inject glog include/library/CMake paths into every engine build route;
- refuse to compile CICERO until a standalone glog compile/link probe succeeds.

Released states, legal-order selection, sibling replay, and scientific interpretation are unchanged.


### v3 native-artifact discovery repair

The second bootstrap still found no importable `pydipcc` module.

The official `dipcc` documentation names the Python interface `pydipcc`, and the repository Makefile sends the build output to `fairdiplomacy/`. Therefore v3 no longer assumes the produced filename.

v3:

- snapshots every native library before and after the build;
- scans all `.so`, `.dylib`, and `.pyd` files under the frozen repository;
- identifies the Python binding by the exported `PyInit_pydipcc` symbol rather than its filename;
- inspects every CMake cache/build directory;
- asks CMake for available targets;
- explicitly builds a `pydipcc` target when present;
- tries both the normal repository build and direct `dipcc/compile.sh` with an explicit output directory;
- records the Python ABI and CMake Python configuration;
- if a binary exports `PyInit_pydipcc`, loads that exact binary directly;
- if no such binary exists, fails with a concise build-target/artifact diagnosis rather than an import-path error.

No scientific/replay logic is changed.


### v2 native-engine bootstrap repair

v1 successfully ran the repository-native `make dipcc` build but then failed to expose `pydipcc.Game` to the notebook interpreter.

The official Makefile places the compiled `pydipcc` extension directly inside `fairdiplomacy/`. v1 added only the repository root to `sys.path`, and it also attempted the build through a separate virtual environment even though the extension must be ABI-compatible with the interpreter that imports it.

v2 changes only the native-engine bootstrap:

- build with the current notebook Python interpreter;
- add both the repository root and `fairdiplomacy/` output directory to the import search path;
- prefer extension files matching the current Python extension ABI;
- import the compiled module as top-level `pydipcc` directly from the official output directory;
- retain `fairdiplomacy.pydipcc` as a secondary route;
- write detailed extension, `file`, `ldd`, Python ABI, and import-error diagnostics if loading still fails.

The released-state bank, reconstruction rules, legal-order target selection, sibling adjudication, and scientific protocol are unchanged.


In [1]:
NOTEBOOK_BUILD = "latent-reservations-notebook11-cicero-native-order-replay-v4"
print("=" * 100)
print(f"NOTEBOOK BUILD: {NOTEBOOK_BUILD}")
print("CICERO native engine reconstruction + legal-order sibling replay")
print("=" * 100)


NOTEBOOK BUILD: latent-reservations-notebook11-cicero-native-order-replay-v4
CICERO native engine reconstruction + legal-order sibling replay


## 1. Dependencies, paths, and Notebook 10 provenance


In [2]:
import copy, hashlib, importlib, importlib.util, inspect, itertools, json, os, re, shutil, subprocess, sys
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

subprocess.run(
    [sys.executable, "-m", "pip", "install", "numpy<2", "pandas>=2,<3", "tqdm>=4.66"],
    check=True,
)

PROJECT_ROOT = Path(os.environ.get("LR_PROJECT_ROOT", "/workspace/latent-reservations")).expanduser().resolve()
NOTEBOOK_SLUG = "11_cicero_native_engine_legal_order_replay"
RUN_ID = os.environ.get("LR_RUN_ID", datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
BASE = PROJECT_ROOT / "notebook_outputs" / NOTEBOOK_SLUG
RUN_DIR = BASE / RUN_ID
MANIFEST_DIR = RUN_DIR / "manifests"
DATA_DIR = RUN_DIR / "data"
ENGINE_DIR = DATA_DIR / "engine"
STATE_DIR = DATA_DIR / "states"
BRANCH_DIR = DATA_DIR / "branches"
TABLE_DIR = RUN_DIR / "results" / "tables"
CICERO_REPO = PROJECT_ROOT / "vendor" / "diplomacy_cicero"

for p in [MANIFEST_DIR, ENGINE_DIR, STATE_DIR, BRANCH_DIR, TABLE_DIR]:
    p.mkdir(parents=True, exist_ok=True)
BASE.mkdir(parents=True, exist_ok=True)
(BASE / "latest_run.json").write_text(json.dumps({
    "notebook_slug": NOTEBOOK_SLUG,
    "run_id": RUN_ID,
    "run_output_dir": str(RUN_DIR),
    "updated_at_utc": datetime.now(timezone.utc).isoformat(),
}, indent=2))

NB10_BASE = PROJECT_ROOT / "notebook_outputs" / "10_cicero_full_press_bootstrap_data_discovery"
nb10_latest = json.loads((NB10_BASE / "latest_run.json").read_text())
NB10_RUN_DIR = Path(nb10_latest["run_output_dir"])
nb10_summary = json.loads((NB10_RUN_DIR / "results" / "tables" / "result_summary.json").read_text())
if not nb10_summary["readiness"]["ready_for_notebook11_engine_integration"]:
    raise RuntimeError("Notebook 10 readiness gate is not green.")

CICERO_COMMIT = nb10_summary["repo_commit"]
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CICERO_REPO, text=True).strip()
if actual != CICERO_COMMIT:
    subprocess.run(["git", "checkout", "--detach", CICERO_COMMIT], cwd=CICERO_REPO, check=True)

tracked = subprocess.check_output(
    ["git", "status", "--porcelain", "--untracked-files=no"],
    cwd=CICERO_REPO,
    text=True,
).strip()
if tracked:
    raise RuntimeError("Frozen CICERO checkout contains tracked modifications.")

print({"run_dir": str(RUN_DIR), "cicero_commit": CICERO_COMMIT})


{'run_dir': '/workspace/latent-reservations/notebook_outputs/11_cicero_native_engine_legal_order_replay/20260814T185224Z', 'cicero_commit': 'e85afeddb34f5b7c1ea0827203b425a0f7e68ead'}


## 2. Inspect and load/build the official native engine


## Native C++ dependency preflight

Verify `glog/logging.h` and `-lglog` before compiling the official engine.

In [3]:
import tempfile

GLOG_LOCAL_SOURCE = PROJECT_ROOT / "vendor" / "google-glog-v0.6.0"
GLOG_LOCAL_PREFIX = PROJECT_ROOT / "vendor" / "google-glog-v0.6.0-install"

def native_cmd(command, *, cwd=None, env=None):
    process = subprocess.run(
        [str(part) for part in command],
        cwd=cwd,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    return {
        "command": [str(part) for part in command],
        "returncode": int(process.returncode),
        "output": process.stdout,
    }

def glog_probe(include_dirs=(), library_dirs=()):
    compiler = shutil.which("c++") or shutil.which("g++")
    if compiler is None:
        return {"ok": False, "error": "No C++ compiler found."}

    with tempfile.TemporaryDirectory(prefix="lr-glog-probe-") as tmp:
        tmp_path = Path(tmp)
        source_path = tmp_path / "probe.cc"
        binary_path = tmp_path / "probe"

        source_path.write_text(
            '#include <glog/logging.h>\n'
            'int main(int argc, char** argv) {\n'
            '  google::InitGoogleLogging(argv[0]);\n'
            '  LOG(INFO) << "probe";\n'
            '  return 0;\n'
            '}\n'
        )

        command = [
            compiler,
            "-std=c++11",
            str(source_path),
            "-o",
            str(binary_path),
        ]

        for path in include_dirs:
            command += ["-I", str(path)]

        for path in library_dirs:
            command += ["-L", str(path)]

        command += ["-lglog"]

        result = native_cmd(command)
        result["ok"] = result["returncode"] == 0
        return result

def try_apt_glog():
    apt_get = shutil.which("apt-get")

    if apt_get is None:
        return {
            "attempted": False,
            "reason": "apt-get unavailable",
        }

    prefix = []

    if os.geteuid() != 0:
        sudo = shutil.which("sudo")

        if sudo is None:
            return {
                "attempted": False,
                "reason": "not root and sudo unavailable",
            }

        sudo_probe = subprocess.run(
            [sudo, "-n", "true"],
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            check=False,
        )

        if sudo_probe.returncode != 0:
            return {
                "attempted": False,
                "reason": "passwordless sudo unavailable",
            }

        prefix = [sudo, "-n"]

    update = native_cmd(prefix + [apt_get, "update"])

    if update["returncode"] != 0:
        return {
            "attempted": True,
            "success": False,
            "stage": "apt-get update",
            "update": update,
        }

    install = native_cmd(
        prefix
        + [
            apt_get,
            "install",
            "-y",
            "libgoogle-glog-dev",
        ]
    )

    return {
        "attempted": True,
        "success": install["returncode"] == 0,
        "update": update,
        "install": install,
    }

def build_local_glog():
    GLOG_LOCAL_SOURCE.parent.mkdir(parents=True, exist_ok=True)

    if (
        GLOG_LOCAL_SOURCE.exists()
        and not (GLOG_LOCAL_SOURCE / ".git").exists()
    ):
        shutil.rmtree(GLOG_LOCAL_SOURCE)

    if not GLOG_LOCAL_SOURCE.exists():
        clone = native_cmd(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "v0.6.0",
                "https://github.com/google/glog.git",
                str(GLOG_LOCAL_SOURCE),
            ]
        )

        if clone["returncode"] != 0:
            return {
                "success": False,
                "stage": "clone",
                "clone": clone,
            }
    else:
        clone = {
            "returncode": 0,
            "output": "Using existing pinned glog source.",
        }

    build_dir = GLOG_LOCAL_SOURCE / "build-lr"
    build_dir.mkdir(parents=True, exist_ok=True)

    configure = native_cmd(
        [
            "cmake",
            "-S",
            str(GLOG_LOCAL_SOURCE),
            "-B",
            str(build_dir),
            "-DCMAKE_BUILD_TYPE=Release",
            f"-DCMAKE_INSTALL_PREFIX={GLOG_LOCAL_PREFIX}",
            "-DBUILD_SHARED_LIBS=ON",
            "-DWITH_GFLAGS=OFF",
            "-DWITH_GTEST=OFF",
            "-DWITH_UNWIND=OFF",
        ]
    )

    if configure["returncode"] != 0:
        return {
            "success": False,
            "stage": "configure",
            "clone": clone,
            "configure": configure,
        }

    build = native_cmd(
        [
            "cmake",
            "--build",
            str(build_dir),
            "--parallel",
            "2",
        ]
    )

    if build["returncode"] != 0:
        return {
            "success": False,
            "stage": "build",
            "clone": clone,
            "configure": configure,
            "build": build,
        }

    install = native_cmd(
        [
            "cmake",
            "--install",
            str(build_dir),
        ]
    )

    return {
        "success": install["returncode"] == 0,
        "stage": "install",
        "clone": clone,
        "configure": configure,
        "build": build,
        "install": install,
        "prefix": str(GLOG_LOCAL_PREFIX),
    }

initial_glog_probe = glog_probe()

GLOG_INCLUDE_DIRS = []
GLOG_LIBRARY_DIRS = []
GLOG_CMAKE_PREFIXES = []
GLOG_DEPENDENCY_ROUTE = None
apt_glog_result = None
local_glog_result = None

if initial_glog_probe["ok"]:
    GLOG_DEPENDENCY_ROUTE = "system_existing"
else:
    apt_glog_result = try_apt_glog()
    after_apt_probe = glog_probe()

    if after_apt_probe["ok"]:
        GLOG_DEPENDENCY_ROUTE = "system_apt"
    else:
        local_glog_result = build_local_glog()

        local_include = GLOG_LOCAL_PREFIX / "include"
        local_library_dirs = [
            path
            for path in [
                GLOG_LOCAL_PREFIX / "lib",
                GLOG_LOCAL_PREFIX / "lib64",
            ]
            if path.exists()
        ]

        local_probe = glog_probe(
            include_dirs=[local_include],
            library_dirs=local_library_dirs,
        )

        if not local_probe["ok"]:
            diagnostics = {
                "initial_probe": initial_glog_probe,
                "apt_result": apt_glog_result,
                "local_build_result": local_glog_result,
                "local_probe": local_probe,
            }

            diagnostics_path = (
                ENGINE_DIR / "glog_dependency_failure.json"
            )
            diagnostics_path.write_text(
                json.dumps(diagnostics, indent=2)
            )

            raise RuntimeError(
                "Could not make glog/logging.h compile before the CICERO build. "
                f"Diagnostics: {diagnostics_path}"
            )

        GLOG_DEPENDENCY_ROUTE = "project_local_glog"
        GLOG_INCLUDE_DIRS = [local_include]
        GLOG_LIBRARY_DIRS = local_library_dirs
        GLOG_CMAKE_PREFIXES = [GLOG_LOCAL_PREFIX]

glog_dependency = {
    "route": GLOG_DEPENDENCY_ROUTE,
    "initial_probe": initial_glog_probe,
    "apt_result": apt_glog_result,
    "local_build_result": local_glog_result,
    "include_dirs": [str(path) for path in GLOG_INCLUDE_DIRS],
    "library_dirs": [str(path) for path in GLOG_LIBRARY_DIRS],
    "cmake_prefixes": [str(path) for path in GLOG_CMAKE_PREFIXES],
}

(ENGINE_DIR / "glog_dependency.json").write_text(
    json.dumps(glog_dependency, indent=2)
)

print(
    json.dumps(
        {
            "glog_dependency_route": GLOG_DEPENDENCY_ROUTE,
            "include_dirs": glog_dependency["include_dirs"],
            "library_dirs": glog_dependency["library_dirs"],
        },
        indent=2,
    )
)


{
  "glog_dependency_route": "system_apt",
  "include_dirs": [],
  "library_dirs": []
}


In [4]:
import importlib.machinery
import sysconfig
import time

MAKEFILE = CICERO_REPO / "Makefile"
DIPCC_DIR = CICERO_REPO / "dipcc"
FAIRDIPLOMACY_DIR = CICERO_REPO / "fairdiplomacy"

for required_path in [
    MAKEFILE,
    DIPCC_DIR,
    FAIRDIPLOMACY_DIR,
    DIPCC_DIR / "compile.sh",
]:
    if not required_path.exists():
        raise FileNotFoundError(
            required_path
        )

make_text = MAKEFILE.read_text(
    errors="replace"
)

if "dipcc:" not in make_text:
    raise RuntimeError(
        "Frozen Makefile does not expose the official dipcc target."
    )

# The official Makefile uses this output directory.
OFFICIAL_PYDIPCC_OUTPUT_DIR = FAIRDIPLOMACY_DIR.resolve()

for import_dir in [
    OFFICIAL_PYDIPCC_OUTPUT_DIR,
    CICERO_REPO.resolve(),
]:
    import_text = str(
        import_dir
    )
    if import_text not in sys.path:
        sys.path.insert(
            0,
            import_text,
        )

NATIVE_SUFFIXES = {
    ".so",
    ".dylib",
    ".pyd",
}

def run_capture(
    command,
    *,
    cwd=None,
    env=None,
    timeout=None,
):
    try:
        proc = subprocess.run(
            command,
            cwd=cwd,
            env=env,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            check=False,
            timeout=timeout,
        )
        return {
            "command": [
                str(
                    part
                )
                for part in command
            ],
            "returncode": int(
                proc.returncode
            ),
            "output": proc.stdout,
        }
    except Exception as exc:
        return {
            "command": [
                str(
                    part
                )
                for part in command
            ],
            "error": repr(
                exc
            ),
            "output": "",
        }

def native_files_under_repo():
    rows = []

    for path in CICERO_REPO.rglob(
        "*"
    ):
        if (
            not path.is_file()
            or path.suffix.lower()
            not in NATIVE_SUFFIXES
        ):
            continue

        try:
            stat = path.stat()
        except OSError:
            continue

        rows.append({
            "path": str(
                path.resolve()
            ),
            "relative_path": str(
                path.relative_to(
                    CICERO_REPO
                )
            ),
            "size_bytes": int(
                stat.st_size
            ),
            "mtime_ns": int(
                stat.st_mtime_ns
            ),
        })

    return sorted(
        rows,
        key=lambda row: row[
            "relative_path"
        ],
    )

def exported_symbols(
    path,
):
    path = Path(
        path
    )

    for command in [
        [
            "nm",
            "-D",
            "--defined-only",
            str(
                path
            ),
        ],
        [
            "objdump",
            "-T",
            str(
                path
            ),
        ],
    ]:
        if shutil.which(
            command[0]
        ) is None:
            continue

        result = run_capture(
            command
        )

        if result.get(
            "returncode"
        ) == 0:
            return result[
                "output"
            ]

    return ""

def has_pydipcc_initializer(
    path,
):
    return (
        "PyInit_pydipcc"
        in exported_symbols(
            path
        )
    )

def native_artifact_diagnostics():
    rows = []

    for row in native_files_under_repo():
        path = Path(
            row[
                "path"
            ]
        )

        symbols = exported_symbols(
            path
        )

        rows.append({
            **row,
            "basename": path.name,
            "contains_dipcc_in_name": (
                "dipcc"
                in path.name.lower()
            ),
            "exports_PyInit_pydipcc": (
                "PyInit_pydipcc"
                in symbols
            ),
            "file": run_capture(
                [
                    "file",
                    str(
                        path
                    ),
                ]
            ),
            "ldd": (
                run_capture(
                    [
                        "ldd",
                        str(
                            path
                        ),
                    ]
                )
                if shutil.which(
                    "ldd"
                )
                else None
            ),
        })

    return rows

def cmake_build_dirs():
    return sorted(
        {
            cache.parent.resolve()
            for cache in CICERO_REPO.rglob(
                "CMakeCache.txt"
            )
            if "dipcc"
            in str(
                cache
            ).lower()
        },
        key=lambda path: str(
            path
        ),
    )

def cmake_cache_extract(
    build_dir,
):
    cache_path = (
        Path(
            build_dir
        )
        / "CMakeCache.txt"
    )

    if not cache_path.exists():
        return {}

    interesting = {}

    for line in cache_path.read_text(
        errors="replace"
    ).splitlines():
        if (
            not line
            or line.startswith(
                "//"
            )
            or line.startswith(
                "#"
            )
            or "=" not in line
        ):
            continue

        lhs, value = line.split(
            "=",
            1,
        )

        name = lhs.split(
            ":",
            1,
        )[
            0
        ]

        if any(
            token
            in name.upper()
            for token in [
                "PYTHON",
                "PYBIND",
                "PYDIP",
                "LIBRARY_OUTPUT",
                "CMAKE_BUILD_TYPE",
                "CMAKE_PROJECT",
            ]
        ):
            interesting[
                name
            ] = value

    return interesting

def cmake_target_inventory(
    build_dir,
):
    result = run_capture(
        [
            "cmake",
            "--build",
            str(
                build_dir
            ),
            "--target",
            "help",
        ]
    )

    text = result.get(
        "output",
        ""
    )

    # Keep likely engine/binding targets only.
    likely = sorted(
        {
            line.strip()
            for line in text.splitlines()
            if (
                "dip"
                in line.lower()
                or "pybind"
                in line.lower()
            )
        }
    )

    return {
        "build_dir": str(
            build_dir
        ),
        "returncode": result.get(
            "returncode"
        ),
        "likely_targets": likely,
        "output_tail": "\n".join(
            text.splitlines()[
                -300:
            ]
        ),
    }

def clear_pydipcc_modules():
    for module_name in [
        "pydipcc",
        "fairdiplomacy.pydipcc",
    ]:
        sys.modules.pop(
            module_name,
            None,
        )

    importlib.invalidate_caches()

def load_binary_as_pydipcc(
    binary_path,
):
    clear_pydipcc_modules()

    binary_path = Path(
        binary_path
    ).resolve()

    loader = importlib.machinery.ExtensionFileLoader(
        "pydipcc",
        str(
            binary_path
        ),
    )

    spec = importlib.util.spec_from_file_location(
        "pydipcc",
        str(
            binary_path
        ),
        loader=loader,
    )

    if (
        spec is None
        or spec.loader is None
    ):
        raise RuntimeError(
            f"Could not create extension spec for {binary_path}"
        )

    module = importlib.util.module_from_spec(
        spec
    )

    sys.modules[
        "pydipcc"
    ] = module

    spec.loader.exec_module(
        module
    )

    return module

def try_normal_imports():
    attempts = []

    clear_pydipcc_modules()

    for module_name in [
        "pydipcc",
        "fairdiplomacy.pydipcc",
    ]:
        try:
            module = importlib.import_module(
                module_name
            )

            if hasattr(
                module,
                "Game",
            ):
                return module, {
                    "route": "normal_import",
                    "module_name": module_name,
                    "module_file": str(
                        Path(
                            module.__file__
                        ).resolve()
                    ),
                    "attempts": attempts,
                }

            attempts.append({
                "route": "normal_import",
                "module_name": module_name,
                "error": "Imported module has no Game.",
                "module_file": str(
                    getattr(
                        module,
                        "__file__",
                        None,
                    )
                ),
            })
        except Exception as exc:
            attempts.append({
                "route": "normal_import",
                "module_name": module_name,
                "error": repr(
                    exc
                ),
            })

    return None, {
        "route": None,
        "attempts": attempts,
    }

def find_initializer_binaries():
    return [
        Path(
            row[
                "path"
            ]
        )
        for row in native_artifact_diagnostics()
        if row[
            "exports_PyInit_pydipcc"
        ]
    ]

def try_initializer_binaries():
    attempts = []

    for path in find_initializer_binaries():
        try:
            module = load_binary_as_pydipcc(
                path
            )

            if hasattr(
                module,
                "Game",
            ):
                return module, {
                    "route": "PyInit_symbol_discovery",
                    "module_name": "pydipcc",
                    "module_file": str(
                        path.resolve()
                    ),
                    "attempts": attempts,
                }

            attempts.append({
                "route": "PyInit_symbol_discovery",
                "path": str(
                    path
                ),
                "error": "Loaded module does not expose Game.",
            })
        except Exception as exc:
            attempts.append({
                "route": "PyInit_symbol_discovery",
                "path": str(
                    path
                ),
                "error": repr(
                    exc
                ),
            })

    return None, {
        "route": None,
        "attempts": attempts,
    }

def base_build_env():
    env = os.environ.copy()

    env["PYDIPCC_OUT_DIR"] = str(OFFICIAL_PYDIPCC_OUTPUT_DIR)
    env["SKIP_TESTS"] = "1"
    env["PYTHON"] = sys.executable
    env["PYTHON_EXECUTABLE"] = sys.executable
    env["Python_EXECUTABLE"] = sys.executable
    env["Python3_EXECUTABLE"] = sys.executable

    prefixes = []

    try:
        pybind11_cmake = subprocess.check_output(
            [sys.executable, "-m", "pybind11", "--cmakedir"],
            text=True,
        ).strip()

        if pybind11_cmake:
            prefixes.append(pybind11_cmake)
    except Exception:
        pass

    prefixes.extend(str(path) for path in GLOG_CMAKE_PREFIXES)

    existing_prefix = env.get("CMAKE_PREFIX_PATH")
    if existing_prefix:
        prefixes.append(existing_prefix)

    if prefixes:
        env["CMAKE_PREFIX_PATH"] = os.pathsep.join(prefixes)

    if GLOG_INCLUDE_DIRS:
        include_paths = [str(path) for path in GLOG_INCLUDE_DIRS]

        for variable in [
            "CPLUS_INCLUDE_PATH",
            "CPATH",
            "CMAKE_INCLUDE_PATH",
        ]:
            existing = env.get(variable)
            values = include_paths + ([existing] if existing else [])
            env[variable] = os.pathsep.join(values)

    if GLOG_LIBRARY_DIRS:
        library_paths = [str(path) for path in GLOG_LIBRARY_DIRS]

        for variable in [
            "LIBRARY_PATH",
            "LD_LIBRARY_PATH",
            "CMAKE_LIBRARY_PATH",
        ]:
            existing = env.get(variable)
            values = library_paths + ([existing] if existing else [])
            env[variable] = os.pathsep.join(values)

    return env
build_started_ns = time.time_ns()
before_native = native_files_under_repo()

# Existing usable artifact first.
pydipcc, import_record = try_normal_imports()

if pydipcc is None:
    pydipcc, symbol_record = try_initializer_binaries()

    if pydipcc is not None:
        import_record = symbol_record
    else:
        import_record[
            "attempts"
        ].extend(
            symbol_record.get(
                "attempts",
                []
            )
        )

build_records = []

if pydipcc is None:
    subprocess.run(
        [
            "git",
            "submodule",
            "update",
            "--init",
            "--recursive",
        ],
        cwd=CICERO_REPO,
        check=True,
    )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--upgrade",
            "pybind11",
            "cmake",
            "ninja",
        ],
        check=True,
    )

    # Route A: exact repository Makefile target.
    make_result = run_capture(
        [
            "make",
            "dipcc",
        ],
        cwd=CICERO_REPO,
        env=base_build_env(),
    )

    build_records.append({
        "route": "make_dipcc",
        **make_result,
    })

    (
        ENGINE_DIR
        / "native_build_make_dipcc.txt"
    ).write_text(
        make_result.get(
            "output",
            ""
        )
    )

    # Route B: invoke the exact official script directly with the output dir
    # passed explicitly. This avoids any shell/make variable propagation ambiguity.
    if not find_initializer_binaries():
        direct_result = run_capture(
            [
                "bash",
                str(
                    DIPCC_DIR
                    / "compile.sh"
                ),
            ],
            cwd=CICERO_REPO,
            env=base_build_env(),
        )

        build_records.append({
            "route": "direct_compile_sh",
            **direct_result,
        })

        (
            ENGINE_DIR
            / "native_build_direct_compile_sh.txt"
        ).write_text(
            direct_result.get(
                "output",
                ""
            )
        )

    # Route C: if CMake configured a pydipcc target, explicitly build it.
    cmake_records = []

    for build_dir in cmake_build_dirs():
        target_info = cmake_target_inventory(
            build_dir
        )
        cmake_records.append({
            **target_info,
            "cache": cmake_cache_extract(
                build_dir
            ),
        })

        target_text = "\n".join(
            target_info.get(
                "likely_targets",
                []
            )
        ).lower()

        if (
            "pydipcc"
            in target_text
            and not find_initializer_binaries()
        ):
            explicit_result = run_capture(
                [
                    "cmake",
                    "--build",
                    str(
                        build_dir
                    ),
                    "--target",
                    "pydipcc",
                    "--",
                    "-j2",
                ],
                env=base_build_env(),
            )

            build_records.append({
                "route": "explicit_cmake_pydipcc_target",
                "build_dir": str(
                    build_dir
                ),
                **explicit_result,
            })

    (
        ENGINE_DIR
        / "cmake_target_inventory.json"
    ).write_text(
        json.dumps(
            cmake_records,
            indent=2,
        )
    )

    # Try again after every build route.
    pydipcc, normal_after = try_normal_imports()

    if pydipcc is not None:
        import_record = normal_after
    else:
        pydipcc, symbol_after = try_initializer_binaries()

        import_record = {
            "route": (
                symbol_after.get(
                    "route"
                )
                if pydipcc is not None
                else None
            ),
            "attempts": (
                import_record.get(
                    "attempts",
                    []
                )
                + normal_after.get(
                    "attempts",
                    []
                )
                + symbol_after.get(
                    "attempts",
                    []
                )
            ),
        }

        if pydipcc is not None:
            import_record.update({
                key: value
                for key, value in symbol_after.items()
                if key != "attempts"
            })

after_native = native_files_under_repo()
artifacts = native_artifact_diagnostics()

new_or_updated_native = [
    row
    for row in after_native
    if row[
        "mtime_ns"
    ]
    >= build_started_ns
]

diagnostic_payload = {
    "python": {
        "executable": sys.executable,
        "version": sys.version,
        "soabi": sysconfig.get_config_var(
            "SOABI"
        ),
        "extension_suffixes": list(
            importlib.machinery.EXTENSION_SUFFIXES
        ),
    },
    "repo": str(
        CICERO_REPO
    ),
    "official_output_dir": str(
        OFFICIAL_PYDIPCC_OUTPUT_DIR
    ),
    "build_records": build_records,
    "cmake_build_dirs": [
        {
            "path": str(
                build_dir
            ),
            "cache": cmake_cache_extract(
                build_dir
            ),
            "targets": cmake_target_inventory(
                build_dir
            ),
        }
        for build_dir in cmake_build_dirs()
    ],
    "native_files_before": before_native,
    "native_files_after": after_native,
    "new_or_updated_native_files": new_or_updated_native,
    "all_native_artifact_diagnostics": artifacts,
    "pydipcc_initializer_binaries": [
        row
        for row in artifacts
        if row[
            "exports_PyInit_pydipcc"
        ]
    ],
    "import_record": import_record,
}

diagnostics_path = (
    ENGINE_DIR
    / "pydipcc_artifact_diagnostics.json"
)

diagnostics_path.write_text(
    json.dumps(
        diagnostic_payload,
        indent=2,
    )
)

if (
    pydipcc is None
    or not hasattr(
        pydipcc,
        "Game",
    )
):
    initializer_rows = diagnostic_payload[
        "pydipcc_initializer_binaries"
    ]

    new_files = diagnostic_payload[
        "new_or_updated_native_files"
    ]

    build_tail = []

    for record in build_records:
        output = record.get(
            "output",
            ""
        )

        if output:
            build_tail.extend(
                output.splitlines()[
                    -60:
                ]
            )

    if initializer_rows:
        diagnosis = (
            "A native binary exporting PyInit_pydipcc exists, "
            "but it is not loadable by this Python interpreter. "
            "This is an ABI/linker compatibility problem rather than a filename/path problem."
        )
    elif new_files:
        diagnosis = (
            "The build produced native libraries, but none exports PyInit_pydipcc. "
            "The Python binding target was not produced even though other native targets built."
        )
    else:
        diagnosis = (
            "The build produced no new native library at all. "
            "The repository build/configuration did not actually compile the Python binding."
        )

    raise RuntimeError(
        diagnosis
        + f"\nFull diagnostics: {diagnostics_path}\n"
        + "Build tail:\n"
        + "\n".join(
            build_tail[
                -120:
            ]
        )
    )

module_file = Path(
    pydipcc.__file__
).resolve()

try:
    module_file.relative_to(
        CICERO_REPO.resolve()
    )
except ValueError:
    raise RuntimeError(
        f"pydipcc resolved outside frozen CICERO checkout: {module_file}"
    )

tracked_after = subprocess.check_output(
    [
        "git",
        "status",
        "--porcelain",
        "--untracked-files=no",
    ],
    cwd=CICERO_REPO,
    text=True,
).strip()

if tracked_after:
    raise RuntimeError(
        "Native build altered tracked CICERO sources."
    )

import_record[
    "commit"
] = CICERO_COMMIT
import_record[
    "module_file"
] = str(
    module_file
)
import_record[
    "python_executable"
] = sys.executable
import_record[
    "python_soabi"
] = sysconfig.get_config_var(
    "SOABI"
)
import_record[
    "artifact_diagnostics"
] = str(
    diagnostics_path
)

(
    ENGINE_DIR
    / "pydipcc_import.json"
).write_text(
    json.dumps(
        import_record,
        indent=2,
    )
)

print(
    json.dumps(
        import_record,
        indent=2,
    )
)


{
  "route": "normal_import",
  "module_name": "pydipcc",
  "module_file": "/workspace/latent-reservations/vendor/diplomacy_cicero/fairdiplomacy/pydipcc.cpython-312-x86_64-linux-gnu.so",
  "attempts": [],
  "commit": "e85afeddb34f5b7c1ea0827203b425a0f7e68ead",
  "python_executable": "/usr/local/bin/python",
  "python_soabi": "cpython-312-x86_64-linux-gnu",
  "artifact_diagnostics": "/workspace/latent-reservations/notebook_outputs/11_cicero_native_engine_legal_order_replay/20260814T185224Z/data/engine/pydipcc_artifact_diagnostics.json"
}


## 3. Verify native runtime interface


In [5]:
Game = pydipcc.Game
required = ["from_json", "get_orderable_locations", "get_all_possible_orders", "set_orders", "process"]
missing = [name for name in required if not hasattr(Game, name)]
if missing:
    raise RuntimeError(f"pydipcc.Game missing required methods: {missing}")

members = sorted(name for name in dir(Game) if not name.startswith("__"))
interesting = [
    name for name in members
    if any(token in name.lower() for token in ["json", "phase", "state", "order", "roll", "message"])
]
runtime_interface = {
    "required_methods": required,
    "interesting_members": interesting,
}
(ENGINE_DIR / "runtime_game_interface.json").write_text(json.dumps(runtime_interface, indent=2))
print(json.dumps(runtime_interface, indent=2))


{
  "required_methods": [
    "from_json",
    "get_orderable_locations",
    "get_all_possible_orders",
    "set_orders",
    "process"
  ],
  "interesting_members": [
    "add_message",
    "clear_old_all_possible_orders",
    "clear_orders",
    "compute_order_history_hash",
    "current_short_phase",
    "delete_message_at_timestamp",
    "from_json",
    "from_json_inplace",
    "get_all_phase_names",
    "get_all_phases",
    "get_all_possible_orders",
    "get_current_phase",
    "get_last_message_timestamp",
    "get_next_phase",
    "get_orderable_locations",
    "get_orders",
    "get_phase_data",
    "get_phase_history",
    "get_prev_phase",
    "get_staged_phase_data",
    "get_state",
    "message_history",
    "messages",
    "phase",
    "phase_of_last_message_at_or_before",
    "phase_type",
    "rollback_messages_to_timestamp_end",
    "rollback_messages_to_timestamp_start",
    "rolled_back_to_phase_end",
    "rolled_back_to_phase_start",
    "rolled_back_to_timestam

## 4. Load the pre-outcome Notebook 10 candidate bank


In [6]:
candidate_df = pd.read_csv(NB10_RUN_DIR / "results" / "tables" / "structural_candidate_states.csv")
candidate_df = candidate_df[candidate_df["state_present"].astype(bool)].copy()
candidate_df = candidate_df.sort_values(["game_index", "phase_index", "power"]).reset_index(drop=True)

print({
    "candidates": len(candidate_df),
    "source_games": int(candidate_df["game_index"].nunique()),
})


{'candidates': 1165, 'source_games': 40}


## 5. Reconstruction helpers


In [7]:
def normalize_nested(value):
    if isinstance(value, dict):
        return {str(k): normalize_nested(v) for k, v in sorted(value.items(), key=lambda kv: str(kv[0]))}
    if isinstance(value, (list, tuple, set, frozenset)):
        vals = [normalize_nested(x) for x in value]
        try:
            return sorted(vals, key=lambda x: json.dumps(x, sort_keys=True))
        except Exception:
            return vals
    if isinstance(value, np.generic):
        return value.item()
    return value

def phase_name_of(game):
    for name in ["current_short_phase", "get_current_phase", "phase"]:
        if not hasattr(game, name):
            continue
        value = getattr(game, name)
        try:
            value = value() if callable(value) else value
        except TypeError:
            continue
        if value is not None:
            return str(value)
    return None

def game_json_text(game):
    if not hasattr(game, "to_json"):
        raise RuntimeError("Native Game has no to_json method.")
    value = game.to_json()
    return value if isinstance(value, str) else json.dumps(value)

def engine_state(game):
    if hasattr(game, "get_state"):
        value = game.get_state()
        if isinstance(value, str):
            try:
                value = json.loads(value)
            except Exception:
                pass
        return normalize_nested(value)
    payload = json.loads(game_json_text(game))
    phases = payload.get("phases", [])
    return normalize_nested(phases[-1].get("state")) if phases else None

def board_signature(state):
    if not isinstance(state, dict):
        return None
    keys = ["name", "units", "centers", "homes", "retreats", "builds"]
    return normalize_nested({k: state.get(k) for k in keys if k in state})

def released_record(row):
    payload = json.loads((CICERO_REPO / str(row["source_path"])).read_text())
    if isinstance(payload, dict) and isinstance(payload.get("phases"), list):
        return payload
    local_id = str(row["local_id"])
    if isinstance(payload, dict) and local_id in payload:
        return payload[local_id]
    if isinstance(payload, list):
        return payload[int(local_id)]
    raise RuntimeError(f"Could not locate released game for {row['candidate_id']}")

rollback_methods = [
    name for name in ["rolled_back_to_phase_start", "rolled_back_to_phase_end", "rolled_back_to_phase"]
    if hasattr(Game, name)
]

def reconstruction_trials(record, phase_index, phase_name):
    trials = []
    try:
        full = Game.from_json(json.dumps(record))
        trials.append(("full", full))
        for method_name in rollback_methods:
            try:
                rolled = getattr(full, method_name)(phase_name)
                if rolled is not None:
                    trials.append((method_name, rolled))
            except Exception:
                pass
    except Exception:
        pass

    prefix = copy.deepcopy(record)
    prefix["phases"] = copy.deepcopy(record["phases"][: phase_index + 1])
    for route, payload in [
        ("prefix", prefix),
        ("prefix_orders_cleared", copy.deepcopy(prefix)),
    ]:
        if route == "prefix_orders_cleared" and payload["phases"]:
            payload["phases"][-1]["orders"] = {}
            payload["phases"][-1].pop("results", None)
            payload["phases"][-1].pop("order_results", None)
        try:
            trials.append((route, Game.from_json(json.dumps(payload))))
        except Exception:
            pass
    return trials


## 6. Reconstruct exact released movement phases


In [8]:
reconstruction_rows = []
accepted = []
good_games = set()
max_attempts = int(os.environ.get("LR_11_MAX_CANDIDATE_ATTEMPTS", "160"))

for _, row in tqdm(
    candidate_df.head(max_attempts).iterrows(),
    total=min(max_attempts, len(candidate_df)),
    desc="Reconstruct released phases",
    unit="candidate",
):
    record = released_record(row)
    phase_index = int(row["phase_index"])
    phase_name = str(row["phase_name"])
    power = str(row["power"])
    released_phase = record["phases"][phase_index]
    released_sig = board_signature(released_phase.get("state"))

    chosen = None
    for route, game in reconstruction_trials(record, phase_index, phase_name):
        observed_phase = phase_name_of(game)
        state_match = board_signature(engine_state(game)) == released_sig
        phase_match = observed_phase == phase_name
        multi = []

        if phase_match and state_match:
            try:
                orderable = game.get_orderable_locations()
                all_orders = game.get_all_possible_orders()
                multi = [
                    loc for loc in orderable.get(power, [])
                    if len(all_orders.get(loc, [])) >= 2
                ]
            except Exception:
                multi = []

        reconstruction_rows.append({
            "candidate_id": row["candidate_id"],
            "game_index": int(row["game_index"]),
            "phase_name": phase_name,
            "power": power,
            "route": route,
            "observed_phase": observed_phase,
            "phase_match": phase_match,
            "state_match": state_match,
            "multi_choice_locations": len(multi),
        })

        if phase_match and state_match and multi:
            chosen = {
                "row": row.to_dict(),
                "record": record,
                "phase": released_phase,
                "game": game,
                "route": route,
                "multi": list(multi),
            }
            break

    if chosen is not None:
        accepted.append(chosen)
        good_games.add(int(row["game_index"]))

    if len(good_games) >= 3:
        break

pd.DataFrame(reconstruction_rows).to_csv(TABLE_DIR / "reconstruction_attempts.csv", index=False)

if len(good_games) < 3:
    raise RuntimeError("Fewer than three released source games reconstructed exactly with multi-choice legal orders.")

print({"accepted_candidates": len(accepted), "source_games": sorted(good_games)})


Reconstruct released phases:   0%|          | 0/160 [00:00<?, ?candidate/s]

{'accepted_candidates': 39, 'source_games': [0, 1, 2]}


## 7. Deterministic legal-order pair scoring


In [9]:
def normalize_messages(raw):
    values = list(raw.values()) if isinstance(raw, dict) else raw if isinstance(raw, list) else []
    out = []
    for item in values:
        if not isinstance(item, dict):
            continue
        sender = item.get("sender") or item.get("from") or item.get("source")
        recipient = item.get("recipient") or item.get("to") or item.get("target")
        text = item.get("message") or item.get("text") or item.get("content")
        if isinstance(text, str) and text.strip():
            out.append({"sender": str(sender), "recipient": str(recipient), "text": text.strip()})
    return out

def order_kind(order):
    tokens = str(order).split()
    if len(tokens) < 3:
        return "OTHER"
    marker = tokens[2]
    return {"H": "HOLD", "-": "MOVE", "S": "SUPPORT", "C": "CONVOY"}.get(marker, "OTHER")

def order_destination(order):
    tokens = str(order).split()
    if "-" in tokens:
        i = tokens.index("-")
        if i + 1 < len(tokens):
            return tokens[i + 1]
    return None

def overlap(order, text):
    toks = {
        token.strip("(),.;:!?[]{}").upper()
        for token in str(order).split()
        if len(token) >= 3
    }
    upper = text.upper()
    return sum(token in upper for token in toks)

def pair_score(a, b, message_text):
    ka, kb = order_kind(a), order_kind(b)
    score = 0
    if "SUPPORT" in {ka, kb} and ka != kb:
        score += 100
    if {ka, kb} == {"MOVE", "HOLD"}:
        score += 80
    if ka == kb == "MOVE" and order_destination(a) != order_destination(b):
        score += 60
    if ka != kb:
        score += 20
    score += 3 * (overlap(a, message_text) + overlap(b, message_text))
    return score


## 8. Identical-parent native sibling adjudication


In [10]:
POWERS = ["AUSTRIA", "ENGLAND", "FRANCE", "GERMANY", "ITALY", "RUSSIA", "TURKEY"]

def order_location(order):
    tokens = str(order).split()
    return tokens[1] if len(tokens) >= 2 else None

def clone_game(game):
    return Game.from_json(game_json_text(game))

def post_hash(game):
    state = board_signature(engine_state(game))
    return hashlib.sha256(
        json.dumps(state, sort_keys=True, ensure_ascii=False, separators=(",", ":")).encode()
    ).hexdigest()

def branch_once(parent, released_orders, focal_power, focal_loc, focal_order):
    game = clone_game(parent)
    possible = game.get_all_possible_orders()
    orderable = game.get_orderable_locations()
    submitted = {}

    for power in POWERS:
        locs = list(orderable.get(power, []))
        if not locs:
            continue
        historical = [str(x) for x in released_orders.get(power, [])]
        by_loc = {order_location(x): x for x in historical if order_location(x) is not None}
        chosen_orders = []

        for loc in locs:
            legal = [str(x) for x in possible.get(loc, [])]
            if not legal:
                continue
            if power == focal_power and loc == focal_loc:
                chosen = str(focal_order)
            else:
                hist = by_loc.get(loc)
                if hist in legal:
                    chosen = hist
                else:
                    holds = [x for x in legal if order_kind(x) == "HOLD"]
                    chosen = holds[0] if holds else legal[0]
            if chosen not in legal:
                raise RuntimeError(f"Non-legal chosen order {chosen!r} for {power} {loc}")
            chosen_orders.append(chosen)

        if chosen_orders:
            game.set_orders(power, chosen_orders)
            submitted[power] = chosen_orders

    game.process()
    return {
        "submitted": submitted,
        "post_hash": post_hash(game),
        "post_phase": phase_name_of(game),
        "post_state": board_signature(engine_state(game)),
    }

pair_attempts = []
selected = None

for candidate in tqdm(accepted, desc="Find divergent legal pair", unit="candidate"):
    row = candidate["row"]
    game = candidate["game"]
    phase = candidate["phase"]
    power = str(row["power"])
    messages = normalize_messages(phase.get("messages"))
    subject_messages = [
        m for m in messages
        if m["sender"].upper() == power or m["recipient"].upper() == power
    ]
    message_text = "\n".join(m["text"] for m in subject_messages)
    possible = game.get_all_possible_orders()

    pairs = []
    for loc in candidate["multi"]:
        legal = sorted(str(x) for x in possible.get(loc, []))
        for a, b in itertools.combinations(legal, 2):
            pairs.append((pair_score(a, b, message_text), str(loc), a, b))
    pairs.sort(key=lambda x: (-x[0], x[1], x[2], x[3]))

    parent_json = game_json_text(game)
    parent_hash = hashlib.sha256(parent_json.encode()).hexdigest()
    released_orders = phase.get("orders", {})

    for score, loc, a, b in pairs:
        try:
            ba = branch_once(game, released_orders, power, loc, a)
            bb = branch_once(game, released_orders, power, loc, b)
            same_parent = hashlib.sha256(game_json_text(game).encode()).hexdigest() == parent_hash
            divergent = ba["post_hash"] != bb["post_hash"]
            pair_attempts.append({
                "candidate_id": row["candidate_id"],
                "location": loc,
                "order_a": a,
                "order_b": b,
                "pair_score": int(score),
                "identical_parent": bool(same_parent),
                "post_states_diverge": bool(divergent),
            })
            if same_parent and divergent:
                selected = {
                    "candidate": candidate,
                    "subject_messages": subject_messages,
                    "location": loc,
                    "order_a": a,
                    "order_b": b,
                    "score": int(score),
                    "parent_json": parent_json,
                    "parent_hash": parent_hash,
                    "branch_a": ba,
                    "branch_b": bb,
                }
                break
        except Exception as exc:
            pair_attempts.append({
                "candidate_id": row["candidate_id"],
                "location": loc,
                "order_a": a,
                "order_b": b,
                "pair_score": int(score),
                "identical_parent": False,
                "post_states_diverge": False,
                "error": repr(exc),
            })
    if selected is not None:
        break

pd.DataFrame(pair_attempts).to_csv(TABLE_DIR / "legal_pair_attempts.csv", index=False)

if selected is None:
    raise RuntimeError("No exact reconstructed parent yielded two legal focal orders with divergent native consequences.")

print({
    "candidate_id": selected["candidate"]["row"]["candidate_id"],
    "location": selected["location"],
    "order_a": selected["order_a"],
    "order_b": selected["order_b"],
    "parent_hash": selected["parent_hash"],
    "post_a": selected["branch_a"]["post_hash"],
    "post_b": selected["branch_b"]["post_hash"],
})


Find divergent legal pair:   0%|          | 0/39 [00:00<?, ?candidate/s]

{'candidate_id': 'cicero_g000_p000_ENGLAND', 'location': 'EDI', 'order_a': 'F EDI - NTH', 'order_b': 'F EDI S F LON - NTH', 'parent_hash': 'e806179f3e70c1f5d300e93fcb6151ad0471d891e6cdcb4a1a713bad4b50423b', 'post_a': '4310eeefea4420a60b4a375a97c06b708afdd4a634a0d12a6c589bf916d00893', 'post_b': 'acab570d11bf5d2e42872b40aba943de0e40ad766c779e1e8f09978680eaae92'}


## 9. Freeze selected parent, branches, and exact next-stage text


In [11]:
candidate = selected["candidate"]
row = candidate["row"]
phase = candidate["phase"]
power = str(row["power"])

parent_record = {
    "candidate_id": row["candidate_id"],
    "source_path": row["source_path"],
    "game_index": int(row["game_index"]),
    "phase_index": int(row["phase_index"]),
    "phase_name": str(row["phase_name"]),
    "power": power,
    "reconstruction_route": candidate["route"],
    "focal_location": selected["location"],
    "legal_choice_a": selected["order_a"],
    "legal_choice_b": selected["order_b"],
    "parent_game_sha256": selected["parent_hash"],
    "released_subject_messages": selected["subject_messages"],
    "released_preorder_state": phase.get("state", {}),
    "historical_orders_metadata": phase.get("orders", {}),
}
(STATE_DIR / "selected_parent.json").write_text(json.dumps(parent_record, indent=2, ensure_ascii=False))
(STATE_DIR / "selected_parent_game.json").write_text(selected["parent_json"])

for name, order, branch in [
    ("A", selected["order_a"], selected["branch_a"]),
    ("B", selected["order_b"], selected["branch_b"]),
]:
    (BRANCH_DIR / f"branch_{name}.json").write_text(json.dumps({
        "branch": name,
        "parent_game_sha256": selected["parent_hash"],
        "focal_power": power,
        "focal_location": selected["location"],
        "focal_order": order,
        "submitted_orders": branch["submitted"],
        "post_phase": branch["post_phase"],
        "post_state_sha256": branch["post_hash"],
        "post_state": branch["post_state"],
    }, indent=2, ensure_ascii=False))

message_lines = [
    f"{m['sender']} -> {m['recipient']}: {m['text']}"
    for m in selected["subject_messages"]
]
parent_text = (
    f"You are controlling {power} in a full-press Diplomacy movement phase.\n"
    f"Phase: {row['phase_name']}\n\n"
    "<PRE_ORDER_BOARD_STATE>\n"
    + json.dumps(phase.get("state", {}), sort_keys=True, ensure_ascii=False)
    + "\n</PRE_ORDER_BOARD_STATE>\n\n"
    "<RELEASED_PRIVATE_MESSAGES_INVOLVING_YOUR_POWER>\n"
    + ("\n".join(message_lines) if message_lines else "(No released focal-power messages.)")
    + "\n</RELEASED_PRIVATE_MESSAGES_INVOLVING_YOUR_POWER>\n\n"
    "The released record may omit historical conversations that were not released. Use only the information shown here.\n"
    f"For unit/location {selected['location']}, the native engine verifies both orders are legal:\n"
    f"OPTION A: {selected['order_a']}\n"
    f"OPTION B: {selected['order_b']}\n"
    "Do not choose an option yet."
)
parent_text_hash = hashlib.sha256(parent_text.encode()).hexdigest()
(STATE_DIR / "selected_parent_text.json").write_text(json.dumps({
    "candidate_id": row["candidate_id"],
    "parent_text_sha256": parent_text_hash,
    "historical_orders_excluded": True,
    "exact_parent_text": parent_text,
}, indent=2, ensure_ascii=False))

print(parent_text)


You are controlling ENGLAND in a full-press Diplomacy movement phase.
Phase: S1901M

<PRE_ORDER_BOARD_STATE>
{"builds": {"AUSTRIA": {"count": 0, "homes": []}, "ENGLAND": {"count": 0, "homes": []}, "FRANCE": {"count": 0, "homes": []}, "GERMANY": {"count": 0, "homes": []}, "ITALY": {"count": 0, "homes": []}, "RUSSIA": {"count": 0, "homes": []}, "TURKEY": {"count": 0, "homes": []}}, "centers": {"AUSTRIA": ["VIE", "TRI", "BUD"], "ENGLAND": ["EDI", "LON", "LVP"], "FRANCE": ["BRE", "PAR", "MAR"], "GERMANY": ["KIE", "MUN", "BER"], "ITALY": ["NAP", "ROM", "VEN"], "RUSSIA": ["STP", "MOS", "WAR", "SEV"], "TURKEY": ["ANK", "SMY", "CON"]}, "homes": {"AUSTRIA": ["BUD", "TRI", "VIE"], "ENGLAND": ["EDI", "LON", "LVP"], "FRANCE": ["BRE", "MAR", "PAR"], "GERMANY": ["BER", "KIE", "MUN"], "ITALY": ["NAP", "ROM", "VEN"], "RUSSIA": ["MOS", "SEV", "STP", "WAR"], "TURKEY": ["ANK", "CON", "SMY"]}, "name": "S1901M", "retreats": {"AUSTRIA": {}, "ENGLAND": {}, "FRANCE": {}, "GERMANY": {}, "ITALY": {}, "RUSSIA": 

## 10. Readiness and final manifest


In [12]:
good_game_ids = sorted({int(c["row"]["game_index"]) for c in accepted})

readiness = {
    "cicero_commit_frozen_clean": True,
    "official_pydipcc_loaded": True,
    "required_engine_methods_present": True,
    "exact_reconstruction_source_games": len(good_game_ids),
    "at_least_three_source_games_reconstructed": len(good_game_ids) >= 3,
    "released_board_state_match": True,
    "legal_focal_orders_enumerated": True,
    "two_legal_choices_selected_preoutcome": True,
    "identical_native_parent": True,
    "native_post_states_diverge": selected["branch_a"]["post_hash"] != selected["branch_b"]["post_hash"],
    "released_focal_messages_preserved": len(selected["subject_messages"]) > 0,
    "historical_orders_excluded_from_subject_text": True,
}
readiness["ready_for_notebook12_full_press_order_pilot"] = all(readiness.values())

(TABLE_DIR / "readiness.json").write_text(json.dumps(readiness, indent=2))

result_summary = {
    "notebook_build": NOTEBOOK_BUILD,
    "run_id": RUN_ID,
    "environment": "CICERO-style full-press Diplomacy",
    "cicero_commit": CICERO_COMMIT,
    "engine_import": import_record,
    "reconstructed_source_games": good_game_ids,
    "selected_candidate": {
        "candidate_id": row["candidate_id"],
        "game_index": int(row["game_index"]),
        "phase_name": str(row["phase_name"]),
        "power": power,
        "focal_location": selected["location"],
        "option_a": selected["order_a"],
        "option_b": selected["order_b"],
        "parent_game_sha256": selected["parent_hash"],
        "post_state_sha256_a": selected["branch_a"]["post_hash"],
        "post_state_sha256_b": selected["branch_b"]["post_hash"],
        "parent_text_sha256": parent_text_hash,
        "released_subject_message_count": len(selected["subject_messages"]),
    },
    "readiness": readiness,
    "interpretation_policy": (
        "Notebook 11 proves native phase reconstruction and divergent legal-order siblings only. "
        "Historical CICERO focal orders are not local-subject labels."
    ),
}
(TABLE_DIR / "result_summary.json").write_text(json.dumps(result_summary, indent=2))

manifest = {
    "notebook_build": NOTEBOOK_BUILD,
    "run_id": RUN_ID,
    "run_output_dir": str(RUN_DIR),
    "source_notebook10_run": str(NB10_RUN_DIR),
    "engine_import": str(ENGINE_DIR / "pydipcc_import.json"),
    "glog_dependency": str(ENGINE_DIR / "glog_dependency.json"),
    "runtime_interface": str(ENGINE_DIR / "runtime_game_interface.json"),
    "reconstruction_attempts": str(TABLE_DIR / "reconstruction_attempts.csv"),
    "legal_pair_attempts": str(TABLE_DIR / "legal_pair_attempts.csv"),
    "selected_parent": str(STATE_DIR / "selected_parent.json"),
    "selected_parent_game": str(STATE_DIR / "selected_parent_game.json"),
    "selected_parent_text": str(STATE_DIR / "selected_parent_text.json"),
    "branch_a": str(BRANCH_DIR / "branch_A.json"),
    "branch_b": str(BRANCH_DIR / "branch_B.json"),
    "readiness": str(TABLE_DIR / "readiness.json"),
    "result_summary": str(TABLE_DIR / "result_summary.json"),
}
(MANIFEST_DIR / "notebook11_manifest.json").write_text(json.dumps(manifest, indent=2))

print(json.dumps(result_summary, indent=2))
print()
print("Notebook 11 complete.")
print(f"Result summary: {TABLE_DIR / 'result_summary.json'}")
print(f"Run directory: {RUN_DIR}")


{
  "notebook_build": "latent-reservations-notebook11-cicero-native-order-replay-v4",
  "run_id": "20260814T185224Z",
  "environment": "CICERO-style full-press Diplomacy",
  "cicero_commit": "e85afeddb34f5b7c1ea0827203b425a0f7e68ead",
  "engine_import": {
    "route": "normal_import",
    "module_name": "pydipcc",
    "module_file": "/workspace/latent-reservations/vendor/diplomacy_cicero/fairdiplomacy/pydipcc.cpython-312-x86_64-linux-gnu.so",
    "attempts": [],
    "commit": "e85afeddb34f5b7c1ea0827203b425a0f7e68ead",
    "python_executable": "/usr/local/bin/python",
    "python_soabi": "cpython-312-x86_64-linux-gnu",
    "artifact_diagnostics": "/workspace/latent-reservations/notebook_outputs/11_cicero_native_engine_legal_order_replay/20260814T185224Z/data/engine/pydipcc_artifact_diagnostics.json"
  },
  "reconstructed_source_games": [
    0,
    1,
    2
  ],
  "selected_candidate": {
    "candidate_id": "cicero_g000_p000_ENGLAND",
    "game_index": 0,
    "phase_name": "S1901M",
  